# Analysis of experiment #1: DPCP heuristics

## Read data

In [76]:
# Read csv

import pandas as pd

df = pd.read_csv("out/stats.csv")

# Some renames to improve readability
df["solver"] = df["solver"].str.replace("v2", "deg")
df["solver"] = df["solver"].str.replace("v3", "edg")

# For the unsolved instances,
# --> write NaN in ub
# --> write NaN in bestTime
# --> write NaN in bestIter
df.loc[df.state != "FEASIBLE", "ub"] = float("nan")
df.loc[df.state != "FEASIBLE", "bestTime"] = float("nan")
df.loc[df.state != "FEASIBLE", "bestIter"] = float("nan")

# Rename some columns
df = df.rename(columns={'ub': 'value', 'state': 'solved'})

# df = df[~df.solver.isin(["heur_semigreedy2s_deg_r1000","heur_semigreedy2s_edg_r1000"])]

df

,instance,solver,run,nvertices,nedges,nA,nB,nvars,ncons,solved,...,otherNodesNColsHeur,otherNodesNColsMwis1,otherNodesNColsMwis2,otherNodesNColsExact,otherNodesTime,otherNodesTimePool,otherNodesTimeHeur,otherNodesTimeMwis1,otherNodesTimeMwis2,otherNodesTimeExact
0,r_n100_p0.25_nA20_nB20_i0,heur_greedy2s_deg,0,100,1186,20,20,-1,-1,FEASIBLE,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,r_n150_p0.75_nA15_nB15_i2,heur_greedy2s_deg,0,150,8450,15,15,-1,-1,FEASIBLE,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,r_n150_p0.25_nA15_nB15_i2,heur_greedy2s_deg,0,150,2737,15,15,-1,-1,FEASIBLE,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,r_n150_p0.75_nA15_nB30_i1,heur_greedy2s_deg,0,150,8396,15,30,-1,-1,FEASIBLE,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
4,r_n150_p0.5_nA15_nB30_i2,heur_greedy2s_deg,0,150,5571,15,30,-1,-1,FEASIBLE,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
931,r_n100_p0.25_nA20_nB10_i0,heur_greedy1s,0,100,1254,20,10,-1,-1,INFEASIBLE,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
932,r_n150_p0.5_nA30_nB15_i1,heur_greedy1s,0,150,5532,30,15,-1,-1,INFEASIBLE,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
933,r_n100_p0.75_nA10_nB20_i1,heur_greedy1s,0,100,3752,10,20,-1,-1,FEASIBLE,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
934,r_n150_p0.25_nA30_nB30_i0,heur_greedy1s,0,150,2776,30,30,-1,-1,FEASIBLE,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


## Semigreedy
### Find the best number of repetitions

In [37]:
# Filter data whose solver is the semigreedy heuristic 
df_sg = df[df.solver.str.contains("semigreedy")]
df_sg

,instance,solver,run,nvertices,nedges,nA,nB,nvars,ncons,solved,...,otherNodesNColsHeur,otherNodesNColsMwis1,otherNodesNColsMwis2,otherNodesNColsExact,otherNodesTime,otherNodesTimePool,otherNodesTimeHeur,otherNodesTimeMwis1,otherNodesTimeMwis2,otherNodesTimeExact
144,r_n100_p0.25_nA20_nB20_i0,heur_semigreedy2s_deg_r100,0,100,1186,20,20,-1,-1,FEASIBLE,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
145,r_n150_p0.75_nA15_nB15_i2,heur_semigreedy2s_deg_r100,0,150,8450,15,15,-1,-1,FEASIBLE,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
146,r_n150_p0.25_nA15_nB15_i2,heur_semigreedy2s_deg_r100,0,150,2737,15,15,-1,-1,FEASIBLE,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
147,r_n150_p0.75_nA15_nB30_i1,heur_semigreedy2s_deg_r100,0,150,8396,15,30,-1,-1,FEASIBLE,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
148,r_n150_p0.5_nA15_nB30_i2,heur_semigreedy2s_deg_r100,0,150,5571,15,30,-1,-1,FEASIBLE,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
859,r_n100_p0.25_nA20_nB10_i0,heur_semigreedy2s_edg_r10000,0,100,1254,20,10,-1,-1,FEASIBLE,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
860,r_n150_p0.5_nA30_nB15_i1,heur_semigreedy2s_edg_r10000,0,150,5532,30,15,-1,-1,FEASIBLE,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
861,r_n100_p0.75_nA10_nB20_i1,heur_semigreedy2s_edg_r10000,0,100,3752,10,20,-1,-1,FEASIBLE,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
862,r_n150_p0.25_nA30_nB30_i0,heur_semigreedy2s_edg_r10000,0,150,2776,30,30,-1,-1,FEASIBLE,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [44]:
# Add a column with the name of the solver but when 10 repetitions are performed
def get_solver10(name):
    tokens = name.split("_")
    tokens[3] = "r10"
    return "_".join(tokens)
df_sg["solver10"] = df_sg.solver.apply(lambda name: get_solver10(name))
df_sg

/tmp/ipykernel_25025/1176879261.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sg["solver10"] = df_sg.solver.apply(lambda name: get_solver10(name))


,instance,solver,run,nvertices,nedges,nA,nB,nvars,ncons,solved,...,otherNodesNColsMwis1,otherNodesNColsMwis2,otherNodesNColsExact,otherNodesTime,otherNodesTimePool,otherNodesTimeHeur,otherNodesTimeMwis1,otherNodesTimeMwis2,otherNodesTimeExact,solver10
144,r_n100_p0.25_nA20_nB20_i0,heur_semigreedy2s_deg_r100,0,100,1186,20,20,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_deg_r10
145,r_n150_p0.75_nA15_nB15_i2,heur_semigreedy2s_deg_r100,0,150,8450,15,15,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_deg_r10
146,r_n150_p0.25_nA15_nB15_i2,heur_semigreedy2s_deg_r100,0,150,2737,15,15,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_deg_r10
147,r_n150_p0.75_nA15_nB30_i1,heur_semigreedy2s_deg_r100,0,150,8396,15,30,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_deg_r10
148,r_n150_p0.5_nA15_nB30_i2,heur_semigreedy2s_deg_r100,0,150,5571,15,30,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_deg_r10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
859,r_n100_p0.25_nA20_nB10_i0,heur_semigreedy2s_edg_r10000,0,100,1254,20,10,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_edg_r10
860,r_n150_p0.5_nA30_nB15_i1,heur_semigreedy2s_edg_r10000,0,150,5532,30,15,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_edg_r10
861,r_n100_p0.75_nA10_nB20_i1,heur_semigreedy2s_edg_r10000,0,100,3752,10,20,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_edg_r10
862,r_n150_p0.25_nA30_nB30_i0,heur_semigreedy2s_edg_r10000,0,150,2776,30,30,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_edg_r10


In [45]:
# Filter data whose solver is the semigreedy heuristic with 10 repetitions
df_sg_r10 = df_sg[df_sg.solver.isin(["heur_semigreedy2s_deg_r10", "heur_semigreedy2s_edg_r10"])]
df_sg_r10

,instance,solver,run,nvertices,nedges,nA,nB,nvars,ncons,solved,...,otherNodesNColsMwis1,otherNodesNColsMwis2,otherNodesNColsExact,otherNodesTime,otherNodesTimePool,otherNodesTimeHeur,otherNodesTimeMwis1,otherNodesTimeMwis2,otherNodesTimeExact,solver10
288,r_n100_p0.25_nA20_nB20_i0,heur_semigreedy2s_deg_r10,0,100,1186,20,20,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_deg_r10
289,r_n150_p0.75_nA15_nB15_i2,heur_semigreedy2s_deg_r10,0,150,8450,15,15,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_deg_r10
290,r_n150_p0.25_nA15_nB15_i2,heur_semigreedy2s_deg_r10,0,150,2737,15,15,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_deg_r10
291,r_n150_p0.75_nA15_nB30_i1,heur_semigreedy2s_deg_r10,0,150,8396,15,30,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_deg_r10
292,r_n150_p0.5_nA15_nB30_i2,heur_semigreedy2s_deg_r10,0,150,5571,15,30,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_deg_r10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
715,r_n100_p0.25_nA20_nB10_i0,heur_semigreedy2s_edg_r10,0,100,1254,20,10,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_edg_r10
716,r_n150_p0.5_nA30_nB15_i1,heur_semigreedy2s_edg_r10,0,150,5532,30,15,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_edg_r10
717,r_n100_p0.75_nA10_nB20_i1,heur_semigreedy2s_edg_r10,0,100,3752,10,20,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_edg_r10
718,r_n150_p0.25_nA30_nB30_i0,heur_semigreedy2s_edg_r10,0,150,2776,30,30,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_edg_r10


In [51]:
# Add a column with the value found by the same solver but when 10 repetitions are performed
df_sg["baseValue"] = df_sg.apply(lambda row: float(df_sg_r10[(df_sg_r10.instance == row.instance) 
                                                       & (df_sg_r10.solver == row.solver10)].value), axis = 1)
df_sg

/tmp/ipykernel_25025/2257227075.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sg["baseValue"] = df_sg.apply(lambda row: float(df_sg_r10[(df_sg_r10.instance == row.instance)


,instance,solver,run,nvertices,nedges,nA,nB,nvars,ncons,solved,...,otherNodesNColsMwis2,otherNodesNColsExact,otherNodesTime,otherNodesTimePool,otherNodesTimeHeur,otherNodesTimeMwis1,otherNodesTimeMwis2,otherNodesTimeExact,solver10,baseValue
144,r_n100_p0.25_nA20_nB20_i0,heur_semigreedy2s_deg_r100,0,100,1186,20,20,-1,-1,FEASIBLE,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_deg_r10,3.0
145,r_n150_p0.75_nA15_nB15_i2,heur_semigreedy2s_deg_r100,0,150,8450,15,15,-1,-1,FEASIBLE,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_deg_r10,6.0
146,r_n150_p0.25_nA15_nB15_i2,heur_semigreedy2s_deg_r100,0,150,2737,15,15,-1,-1,FEASIBLE,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_deg_r10,2.0
147,r_n150_p0.75_nA15_nB30_i1,heur_semigreedy2s_deg_r100,0,150,8396,15,30,-1,-1,FEASIBLE,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_deg_r10,6.0
148,r_n150_p0.5_nA15_nB30_i2,heur_semigreedy2s_deg_r100,0,150,5571,15,30,-1,-1,FEASIBLE,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_deg_r10,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
859,r_n100_p0.25_nA20_nB10_i0,heur_semigreedy2s_edg_r10000,0,100,1254,20,10,-1,-1,FEASIBLE,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_edg_r10,3.0
860,r_n150_p0.5_nA30_nB15_i1,heur_semigreedy2s_edg_r10000,0,150,5532,30,15,-1,-1,FEASIBLE,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_edg_r10,7.0
861,r_n100_p0.75_nA10_nB20_i1,heur_semigreedy2s_edg_r10000,0,100,3752,10,20,-1,-1,FEASIBLE,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_edg_r10,3.0
862,r_n150_p0.25_nA30_nB30_i0,heur_semigreedy2s_edg_r10000,0,150,2776,30,30,-1,-1,FEASIBLE,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_edg_r10,4.0


In [57]:
# Add a column with the improvement (%) in the value with respect to the value found 
# by the same solver but when 10 repetitions are performed
df_sg["improvement"] = df_sg.apply(lambda row: 100 - (row.value * 100 / row.baseValue), axis = 1)
df_sg

/tmp/ipykernel_25025/132978244.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sg["improvement"] = df_sg.apply(lambda row: 100 - (row.value * 100 / row.baseValue), axis = 1)


,instance,solver,run,nvertices,nedges,nA,nB,nvars,ncons,solved,...,otherNodesNColsExact,otherNodesTime,otherNodesTimePool,otherNodesTimeHeur,otherNodesTimeMwis1,otherNodesTimeMwis2,otherNodesTimeExact,solver10,baseValue,improvement
144,r_n100_p0.25_nA20_nB20_i0,heur_semigreedy2s_deg_r100,0,100,1186,20,20,-1,-1,FEASIBLE,...,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_deg_r10,3.0,0.000000
145,r_n150_p0.75_nA15_nB15_i2,heur_semigreedy2s_deg_r100,0,150,8450,15,15,-1,-1,FEASIBLE,...,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_deg_r10,6.0,16.666667
146,r_n150_p0.25_nA15_nB15_i2,heur_semigreedy2s_deg_r100,0,150,2737,15,15,-1,-1,FEASIBLE,...,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_deg_r10,2.0,0.000000
147,r_n150_p0.75_nA15_nB30_i1,heur_semigreedy2s_deg_r100,0,150,8396,15,30,-1,-1,FEASIBLE,...,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_deg_r10,6.0,16.666667
148,r_n150_p0.5_nA15_nB30_i2,heur_semigreedy2s_deg_r100,0,150,5571,15,30,-1,-1,FEASIBLE,...,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_deg_r10,3.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
859,r_n100_p0.25_nA20_nB10_i0,heur_semigreedy2s_edg_r10000,0,100,1254,20,10,-1,-1,FEASIBLE,...,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_edg_r10,3.0,33.333333
860,r_n150_p0.5_nA30_nB15_i1,heur_semigreedy2s_edg_r10000,0,150,5532,30,15,-1,-1,FEASIBLE,...,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_edg_r10,7.0,14.285714
861,r_n100_p0.75_nA10_nB20_i1,heur_semigreedy2s_edg_r10000,0,100,3752,10,20,-1,-1,FEASIBLE,...,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_edg_r10,3.0,0.000000
862,r_n150_p0.25_nA30_nB30_i0,heur_semigreedy2s_edg_r10000,0,150,2776,30,30,-1,-1,FEASIBLE,...,0,0.0,0.0,0.0,0.0,0.0,0.0,heur_semigreedy2s_edg_r10,4.0,25.000000


In [86]:
## TODO: que hacemos con los nan (value.isna())

In [93]:
df_sg2 = df_sg.groupby(["nvertices","solver"]).agg(
    {
        "solved": [("count", lambda x: (x == "FEASIBLE").sum())],
        "improvement": [("avg (%)", "mean")],
        "time": [("avg", "mean")],
        "bestIter": [("avg", "mean")]
    }
).reset_index()
df_sg2

,nvertices,solver,solved,improvement,time,bestIter
,,,count,avg (%),avg,avg
0,100,heur_semigreedy2s_deg_r10,35,0.000000,0.006598,1.400000
1,100,heur_semigreedy2s_deg_r100,36,11.851474,0.054869,18.527778
2,100,heur_semigreedy2s_deg_r1000,36,14.672336,0.521556,57.916667
3,100,heur_semigreedy2s_deg_r10000,36,16.509070,4.895154,293.500000
4,100,heur_semigreedy2s_deg_r100000,36,18.890023,50.871950,3445.638889
5,100,heur_semigreedy2s_edg_r10,35,0.000000,0.004234,2.457143
6,100,heur_semigreedy2s_edg_r100,36,6.952381,0.037503,16.111111
7,100,heur_semigreedy2s_edg_r1000,36,9.812925,0.324866,55.722222
8,100,heur_semigreedy2s_edg_r10000,36,10.384354,3.178106,194.444444


## Best 

In [70]:
# Pivot dataframe
df2 = df.pivot(index="instance", columns="solver", values=["value","bestTime"])
df2

value                                      \
solver                    heur_greedy1s heur_greedy2s_deg heur_greedy2s_edg   
instance                                                                      
r_n100_p0.25_nA10_nB10_i0           2.0               3.0               2.0   
r_n100_p0.25_nA10_nB10_i1           2.0               2.0               2.0   
r_n100_p0.25_nA10_nB10_i2           1.0               3.0               1.0   
r_n100_p0.25_nA10_nB20_i0           2.0               2.0               2.0   
r_n100_p0.25_nA10_nB20_i1           2.0               2.0               2.0   
...                                 ...               ...               ...   
r_n150_p0.75_nA30_nB15_i1           NaN              12.0              10.0   
r_n150_p0.75_nA30_nB15_i2           NaN               NaN               NaN   
r_n150_p0.75_nA30_nB30_i0           NaN              12.0              12.0   
r_n150_p0.75_nA30_nB30_i1          12.0              12.0              11.0   
r_n150_p0.75_nA30_nB30_i2           NaN              10.0              11.0   

                                                     \
solver                    heur_semigreedy2s_deg_r10   
instance                                              
r_n100_p0.25_nA10_nB10_i0                       2.0   
r_n100_p0.25_nA10_nB10_i1                       2.0   
r_n100_p0.25_nA10_nB10_i2                       2.0   
r_n100_p0.25_nA10_nB20_i0                       2.0   
r_n100_p0.25_nA10_nB20_i1                       2.0   
...                                             ...   
r_n150_p0.75_nA30_nB15_i1                       NaN   
r_n150_p0.75_nA30_nB15_i2                      13.0   
r_n150_p0.75_nA30_nB30_i0                      10.0   
r_n150_p0.75_nA30_nB30_i1                      10.0   
r_n150_p0.75_nA30_nB30_i2                       9.0   

                                                      \
solver                    heur_semigreedy2s_deg_r100   
instance                                               
r_n100_p0.25_nA10_nB10_i0                        2.0   
r_n100_p0.25_nA10_nB10_i1                        1.0   
r_n100_p0.25_nA10_nB10_i2                        1.0   
r_n100_p0.25_nA10_nB20_i0                        2.0   
r_n100_p0.25_nA10_nB20_i1                        2.0   
...                                              ...   
r_n150_p0.75_nA30_nB15_i1                       11.0   
r_n150_p0.75_nA30_nB15_i2                       12.0   
r_n150_p0.75_nA30_nB30_i0                       10.0   
r_n150_p0.75_nA30_nB30_i1                        9.0   
r_n150_p0.75_nA30_nB30_i2                        9.0   

                                                       \
solver                    heur_semigreedy2s_deg_r1000   
instance                                                
r_n100_p0.25_nA10_nB10_i0                         2.0   
r_n100_p0.25_nA10_nB10_i1                         1.0   
r_n100_p0.25_nA10_nB10_i2                         1.0   
r_n100_p0.25_nA10_nB20_i0                         2.0   
r_n100_p0.25_nA10_nB20_i1                         2.0   
...                                               ...   
r_n150_p0.75_nA30_nB15_i1                        11.0   
r_n150_p0.75_nA30_nB15_i2                        12.0   
r_n150_p0.75_nA30_nB30_i0                         8.0   
r_n150_p0.75_nA30_nB30_i1                         9.0   
r_n150_p0.75_nA30_nB30_i2                         8.0   

                                                        \
solver                    heur_semigreedy2s_deg_r10000   
instance                                                 
r_n100_p0.25_nA10_nB10_i0                          1.0   
r_n100_p0.25_nA10_nB10_i1                          1.0   
r_n100_p0.25_nA10_nB10_i2                          1.0   
r_n100_p0.25_nA10_nB20_i0                          2.0   
r_n100_p0.25_nA10_nB20_i1                          2.0   
...                                                ...   
r_n150_p0.75_nA30_nB15_i1                         

In [71]:
# Comparison function
def get_winner_heur(row):
    best_heur = None
    min_value = float("inf")
    min_time = float("inf")
    n_heurs = int(len(row)/2)
    for i in range(n_heurs):
        h = df2.columns[i][1] # get heuristic's name
        j = ("bestTime", h) # get column's name of the bestTime for h
        if (row.iloc[i] < min_value) or (row.iloc[i] == min_value and row[j] < min_time):
            best_heur = h
            min_value = row.iloc[i]
            min_time = row[j]
    return best_heur

In [72]:
df2["winner"] = df2.apply(get_winner_heur, axis=1)
df2

value                                      \
solver                    heur_greedy1s heur_greedy2s_deg heur_greedy2s_edg   
instance                                                                      
r_n100_p0.25_nA10_nB10_i0           2.0               3.0               2.0   
r_n100_p0.25_nA10_nB10_i1           2.0               2.0               2.0   
r_n100_p0.25_nA10_nB10_i2           1.0               3.0               1.0   
r_n100_p0.25_nA10_nB20_i0           2.0               2.0               2.0   
r_n100_p0.25_nA10_nB20_i1           2.0               2.0               2.0   
...                                 ...               ...               ...   
r_n150_p0.75_nA30_nB15_i1           NaN              12.0              10.0   
r_n150_p0.75_nA30_nB15_i2           NaN               NaN               NaN   
r_n150_p0.75_nA30_nB30_i0           NaN              12.0              12.0   
r_n150_p0.75_nA30_nB30_i1          12.0              12.0              11.0   
r_n150_p0.75_nA30_nB30_i2           NaN              10.0              11.0   

                                                     \
solver                    heur_semigreedy2s_deg_r10   
instance                                              
r_n100_p0.25_nA10_nB10_i0                       2.0   
r_n100_p0.25_nA10_nB10_i1                       2.0   
r_n100_p0.25_nA10_nB10_i2                       2.0   
r_n100_p0.25_nA10_nB20_i0                       2.0   
r_n100_p0.25_nA10_nB20_i1                       2.0   
...                                             ...   
r_n150_p0.75_nA30_nB15_i1                       NaN   
r_n150_p0.75_nA30_nB15_i2                      13.0   
r_n150_p0.75_nA30_nB30_i0                      10.0   
r_n150_p0.75_nA30_nB30_i1                      10.0   
r_n150_p0.75_nA30_nB30_i2                       9.0   

                                                      \
solver                    heur_semigreedy2s_deg_r100   
instance                                               
r_n100_p0.25_nA10_nB10_i0                        2.0   
r_n100_p0.25_nA10_nB10_i1                        1.0   
r_n100_p0.25_nA10_nB10_i2                        1.0   
r_n100_p0.25_nA10_nB20_i0                        2.0   
r_n100_p0.25_nA10_nB20_i1                        2.0   
...                                              ...   
r_n150_p0.75_nA30_nB15_i1                       11.0   
r_n150_p0.75_nA30_nB15_i2                       12.0   
r_n150_p0.75_nA30_nB30_i0                       10.0   
r_n150_p0.75_nA30_nB30_i1                        9.0   
r_n150_p0.75_nA30_nB30_i2                        9.0   

                                                       \
solver                    heur_semigreedy2s_deg_r1000   
instance                                                
r_n100_p0.25_nA10_nB10_i0                         2.0   
r_n100_p0.25_nA10_nB10_i1                         1.0   
r_n100_p0.25_nA10_nB10_i2                         1.0   
r_n100_p0.25_nA10_nB20_i0                         2.0   
r_n100_p0.25_nA10_nB20_i1                         2.0   
...                                               ...   
r_n150_p0.75_nA30_nB15_i1                        11.0   
r_n150_p0.75_nA30_nB15_i2                        12.0   
r_n150_p0.75_nA30_nB30_i0                         8.0   
r_n150_p0.75_nA30_nB30_i1                         9.0   
r_n150_p0.75_nA30_nB30_i2                         8.0   

                                                        \
solver                    heur_semigreedy2s_deg_r10000   
instance                                                 
r_n100_p0.25_nA10_nB10_i0                          1.0   
r_n100_p0.25_nA10_nB10_i1                          1.0   
r_n100_p0.25_nA10_nB10_i2                          1.0   
r_n100_p0.25_nA10_nB20_i0                          2.0   
r_n100_p0.25_nA10_nB20_i1                          2.0   
...                                                ...   
r_n150_p0.75_nA30_nB15_i1                         

In [73]:
df["winner"] = df.apply(lambda row: row.solver == df2.loc[row.instance].winner, axis=1)
df

,instance,solver,run,nvertices,nedges,nA,nB,nvars,ncons,solved,...,otherNodesNColsMwis1,otherNodesNColsMwis2,otherNodesNColsExact,otherNodesTime,otherNodesTimePool,otherNodesTimeHeur,otherNodesTimeMwis1,otherNodesTimeMwis2,otherNodesTimeExact,winner
0,r_n100_p0.25_nA20_nB20_i0,heur_greedy2s_deg,0,100,1186,20,20,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,False
1,r_n150_p0.75_nA15_nB15_i2,heur_greedy2s_deg,0,150,8450,15,15,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,False
2,r_n150_p0.25_nA15_nB15_i2,heur_greedy2s_deg,0,150,2737,15,15,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,False
3,r_n150_p0.75_nA15_nB30_i1,heur_greedy2s_deg,0,150,8396,15,30,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,False
4,r_n150_p0.5_nA15_nB30_i2,heur_greedy2s_deg,0,150,5571,15,30,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
931,r_n100_p0.25_nA20_nB10_i0,heur_greedy1s,0,100,1254,20,10,-1,-1,INFEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,False
932,r_n150_p0.5_nA30_nB15_i1,heur_greedy1s,0,150,5532,30,15,-1,-1,INFEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,False
933,r_n100_p0.75_nA10_nB20_i1,heur_greedy1s,0,100,3752,10,20,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,False
934,r_n150_p0.25_nA30_nB30_i0,heur_greedy1s,0,150,2776,30,30,-1,-1,FEASIBLE,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,False


## Summary

In [74]:
df3 = df.groupby(["nvertices","solver"]).agg(
    {
        "solved": [("count", lambda x: (x == "FEASIBLE").sum())],
        "winner": [("%", lambda x: (x == True).sum() / len(x) * 100)],
        "value": [("avg", "mean")],
        "time": [("avg", "mean")],
        "bestIter": [("avg", "mean")]
    }
).reset_index()
df3

,nvertices,solver,solved,winner,value,time,bestIter
,,,count,%,avg,avg,avg
0,100,heur_greedy1s,28,2.777778,3.285714,0.003296,0.000000
1,100,heur_greedy2s_deg,36,2.777778,4.638889,0.001235,0.000000
2,100,heur_greedy2s_edg,35,5.555556,4.200000,0.000891,0.000000
3,100,heur_semigreedy2s_deg_r10,35,2.777778,4.000000,0.006598,1.400000
4,100,heur_semigreedy2s_deg_r100,36,0.000000,3.638889,0.054869,18.527778
5,100,heur_semigreedy2s_deg_r1000,36,0.000000,3.500000,0.521556,57.916667
6,100,heur_semigreedy2s_deg_r10000,36,8.333333,3.444444,4.895154,293.500000
7,100,heur_semigreedy2s_deg_r100000,36,0.000000,3.361111,50.871950,3445.638889
8,100,heur_semigreedy2s_edg_r10,35,8.333333,3.485714,0.004234,2.457143
